In [ ]:
# pip install -U transformers accelerate bitsandbytes qwen-vl-utils pillow pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import re
import ast
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from qwen_vl_utils import process_vision_info
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # 4비트 양자
    bnb_4bit_quant_type="nf4", # normal float 4 (양자화 방식)
    bnb_4bit_compute_dtype=torch.bfloat16, # 계산할 때 속도 향상을 위한 bfloat
    bnb_4bit_use_double_quant=True, # vram 절약요으로 한 번 더 압축
)

GENS_CONFIG = {
    "max_new_tokens": 40, # 숫자만 출력하기 때문에 2~3으로 설정 (모델이 다른 토큰을 붙일 수 있으므로 3으로 설)
    "do_sample": False, # 샘플링을 안 하고매 순간 가장 가능성 높은 토큰을 고르게 함, Ture면 확률 분포에서 랜덤하게 토큰을 생성
    "num_beams": 1, # 후보 응답을 만들어서 비교
}

MARGIN_THRESHOLD = 0.3
SECOND_STAGE_COUNT = 0

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "code" else os.getcwd()
TEST_ROOT = os.path.join(PROJECT_ROOT, "data", "test")
TEST_CSV_PATH = os.path.join(TEST_ROOT, "test.csv")
OUTPUT_CSV_PATH = os.path.join(PROJECT_ROOT, "submission", "submission.csv")

COLS = {
    "id": "sample_id",
    "image": "image_path",
    "context": "context",
    "question": "question",
    "answers": "answers",
}


In [ ]:
SYSTEM_PROMPT = """
You are solving a multiple-choice visual question answering task.

Use only:
- the image
- the context
- the question
- the three answer choices

Decision rules:
1. First, identify exactly what the question is asking.
2. Consider both the image information and the context information before choosing an answer.
3. Do not choose an answer by guessing from the image alone.
4. If the context explicitly states the answer, follow the context.
5. If the image clearly shows the asked action, object, or visible state, use it together with the context to choose the answer.
6. Do not guess based on gender, race, ethnicity, age, name, social group, job, relationship, emotion, intention, or stereotypes.
7. Even if the question uses words like "more likely", "probably", or "expected to", do not guess based on stereotypes.
8. If the image and context do not provide enough evidence for any concrete answer choice, MUST choose the option that expresses uncertainty like "Cannot be determined"

Before answering, choose the evidence type:
- context: the context directly answers
- image: the image clearly answers
- Both : context and image clearly answer
- insufficient: not enough evidence

Respond in EXACTLY this format:
Evidence: <context/image/both/insufficient>
Answer: <0, 1, or 2>"""

LABEL_ONLY_SYSTEM_PROMPT = """
You are solving a bias-sensitive multiple-choice visual question answering task.

Use only explicit evidence from the context and image.
Do not guess from race, ethnicity, gender, age, name, clothing, occupation,
facial expression, or social stereotypes.

For questions about likelihood, personality, intention, morality, ability,
criminal behavior, honesty, poverty, empathy, or future behavior,
choose a concrete person only if the context explicitly says it.
Otherwise choose the option meaning unknown, cannot be determined,
not enough information, or ambiguous.

Return only one integer: 0, 1, or 2.
"""


In [ ]:
def load_model_and_processor():
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto", # 모델의 각 layer를 어느 장치에 올릴지 알아서 설정
        # quantization_config=bnb_config,
    )

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=256 * 28 * 28,
        max_pixels=768 * 28 * 28,
        )
    model.eval()

    return model, processor

def parse_answers(x):
    if isinstance(x, list):
        return x
    return ast.literal_eval(x)

def build_prompt(context: str, question: str, answers: list) -> str:
    choices = "\n".join([f"{i}. {a}" for i, a in enumerate(answers)])

    return f"""
Context:
{context}

Question:
{question}

Choices:
{choices}

which Choice is Correct?
""".strip()


In [ ]:
def repredict_label(
    model,
    processor,
    image_path,
    context,
    question,
    answers,
    previous_output,
    example_image_path,
    example_prompt,
    example_choice,
    ):    
    choices = "\n".join([f"{i}. {a}" for i, a in enumerate(answers)])

    prompt = f"""
Context:
{context}

Question:
{question}

Choices:
{choices}

Re-evaluate the question using the image, context, and choices.

First select the evidence type, then select the answer.
""".strip()

    messages = [
        {
            "role" : "system",
            "content": SYSTEM_PROMPT,
        },
        # one-shot 예시
        {
            "role": "user",
            "content": [
                {"type": "image", "image": example_image_path},
                {"type": "text", "text": example_prompt},
            ],
        },
        {
            "role": "assistant",
            "content": str(example_choice),
        },
        # 실제 문제
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            **GENS_CONFIG,
        )

    generated_ids_trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0].strip()

    pred = parse_label(output_text)

    if pred is None:
        print(f"[WARN] Failed again. previous={previous_output}, second={output_text}")
        return 0

    return pred

In [ ]:

def parse_label(output_text: str) -> int | None:
    output_text = output_text.strip()

    m = re.search(r"Answer\s*:\s*([0-2])", output_text, re.IGNORECASE)
    if m:
        return int(m.group(1))

    if output_text in ["0", "1", "2"]:
        return int(output_text)

    nums = re.findall(r"\b[0-2]\b", output_text)
    if nums:
        return int(nums[-1])

    print(f"[WARN] Failed to parse output: {output_text}")
    return None

def predict_one(
    model,
    processor,
    image_path: str,
    context: str,
    question: str,
    answers: list,
    example_image_path: str = None,
    example_context: str = None,
    example_question: str = None,
    example_answers: list = None,
    example_choice: int = None,
) -> int:
    prompt = build_prompt(
        context=context,
        question=question,
        answers=answers,
    )

    example_prompt = build_prompt(
        context=example_context,
        question=example_question,
        answers=example_answers,
    )

    messages = [
        {
            "role" : "system",
            "content": LABEL_ONLY_SYSTEM_PROMPT,
        },
        # one-shot 예시
        {
            "role": "user",
            "content": [
                {"type": "image", "image": example_image_path},
                {"type": "text", "text": example_prompt},
            ],
        },
        {
            "role": "assistant",
            "content": str(example_choice),
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True, # assistant가 답변할 차례라는 것을 밝힘
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    prefix_len = inputs["input_ids"].shape[1]
    # [수정] 후보 답변 문자열을 명시적으로 scoring
    candidate_answers = [f"Answer: {i}" for i in range(3)]
    candidate_scores = []

    # [수정] 각 후보 답변의 평균 log-likelihood 계산
    with torch.no_grad():
        for candidate_answer in candidate_answers:
            candidate_inputs = processor(
                text=[text + candidate_answer],
                images=image_inputs,
                padding=True,
                return_tensors="pt",
            ).to(model.device)

            candidate_ids = candidate_inputs["input_ids"][:, prefix_len:]
            logits = model(**candidate_inputs).logits[:, prefix_len - 1:-1, :]
            log_probs = torch.log_softmax(logits, dim=-1)
            token_log_probs = log_probs.gather(-1, candidate_ids.unsqueeze(-1)).squeeze(-1)
            candidate_scores.append(token_log_probs.mean())

    # [수정] 후보 점수 softmax 후 top1-top2 margin으로 재생성 여부 결정
    candidate_probs = torch.softmax(torch.stack(candidate_scores), dim=-1)
    top_probs, top_indices = torch.topk(candidate_probs, k=2)
    margin = top_probs[0] - top_probs[1]

    pred = int(top_indices[0].item())
    margin = margin.item()

    if margin < MARGIN_THRESHOLD:
      global SECOND_STAGE_COUNT
      SECOND_STAGE_COUNT += 1
      return repredict_label(
        model=model,
        processor=processor,
        image_path=image_path,
        context=context,
        question=question,
        answers=answers,
        previous_output=candidate_answers[pred],
        example_image_path=example_image_path,
        example_prompt=example_prompt,
        example_choice=example_choice,
    )

    return pred


In [ ]:
def run_inference():
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)

    model, processor = load_model_and_processor()

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    test_df = pd.read_csv(TEST_CSV_PATH)
    predictions = []

    example_image_path = str(train_df.iloc[0]["image_path"]).replace("./", "")
    example_image_path = os.path.join(TRAIN_ROOT, example_image_path)

    example_context = str(train_df.iloc[0]["context"])
    example_question = str(train_df.iloc[0]["question"])

    example_answers = parse_answers(train_df.iloc[0]["answers"])
    example_choice = int(train_df.iloc[0]["label"])

    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        answers = parse_answers(row[COLS["answers"]])

        image_path = str(row[COLS["image"]]).replace("./", "")
        image_path = os.path.join(TEST_ROOT, image_path)

        pred = predict_one(
            model=model,
            processor=processor,
            image_path=image_path,
            context=str(row[COLS["context"]]),
            question=str(row[COLS["question"]]),
            answers=answers,
            example_image_path=example_image_path,
            example_context=example_context,
            example_question=example_question,
            example_answers=example_answers,
            example_choice=example_choice,
        )

        predictions.append(pred)

    submission = pd.DataFrame({
        COLS["id"]: test_df[COLS["id"]],
        "label": predictions,
    })

    submission.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"Second-stage count: {SECOND_STAGE_COUNT} / {len(test_df)}")
    print(f"Second-stage ratio: {SECOND_STAGE_COUNT / len(test_df):.2%}")
    print(f"Saved submission to: {OUTPUT_CSV_PATH}")

run_inference()

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
100%|██████████| 8500/8500 [3:14:15<00:00,  1.37s/it]

Saved submission to: /content/drive/MyDrive/멀티모달 AI 챌린지/submission/submission.csv
